# Reproduce PhysAgg-R (notebook conductor + CLI)

Pattern **B + A**:

- **B (this notebook):** parameters, orchestration, and result tables.
- **A (CLI):** heavy training stays in `python -m src.train.physagg`.

Pipeline:

1. Build the physical graph from the released parquet (`src.graphs.build`).
2. Launch paper recipes via subprocess (same flags as the README CLI).
3. Load `experiments/outputs/.../summary.json` and metric files into tables.

Set `RUN_TRAINING = True` locally to train. The published notebook keeps it
`False` so GitHub viewers see the workflow without a multi-hour job.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Flip to True on your machine to actually train (PyTorch Geometric required).
RUN_TRAINING = False

DATA = ROOT / "data" / "datos_wifi_curated_anon.parquet"
GRAPH_DIR = ROOT / "data" / "graphs" / "physical"
OUT_TEMPORAL = ROOT / "experiments" / "outputs" / "physagg_power" / "temporal_noniso"
OUT_SPATIAL = ROOT / "experiments" / "outputs" / "physagg_power" / "spatial_suchiapa"

print("Package root name:", ROOT.name)
print("RUN_TRAINING:", RUN_TRAINING)
print("Data:", DATA.relative_to(ROOT).as_posix(), "| exists:", DATA.exists())

Package root name: 02_reproducibility_github
RUN_TRAINING: False
Data: data/datos_wifi_curated_anon.parquet | exists: True


## 1. Build physical graph (same code as CLI)

Equivalent CLI:

```bash
python -m src.graphs.build --input data/datos_wifi_curated_anon.parquet --output data/graphs
```

PhysAgg only needs the **physical** graph; we build that artefact here.

In [2]:
from src.graphs.build import build_physical_graph, save_graph

df = pd.read_parquet(DATA)
nodes, edges = build_physical_graph(df, nearby_threshold_m=30.0, min_co_observations=2)
stats = save_graph(ROOT / "data" / "graphs", "physical", nodes, edges)
print("Graph stats:", {k: stats[k] for k in stats if k != "name"})
print("Saved under:", "data/graphs/physical/")

Graph stats: {'n_nodes': 36010, 'n_edges': 20074, 'n_components': 31315, 'avg_degree': 1.1149125242988058, 'max_degree': 71}
Saved under: data/graphs/physical/


## 2. Paper recipes (CLI under the hood)

| Recipe | What it matches in the paper |
|:---|:---|
| `temporal_ablation` | Table ablation on temporal non-isolate (5 seeds) |
| `spatial_holdout` | Municipality hold-out (Suchiapa / robust) |

In [3]:
RECIPES = {
    "temporal_ablation": {
        "output": OUT_TEMPORAL,
        "args": [
            "--run", "all",
            "--seeds", "42,0,1,2,3",
            "--split", "temporal",
            "--subgraph", "non_isolate",
            "--mcnemar", "A2e,A0",
            "--with-rf",
        ],
    },
    "spatial_holdout": {
        "output": OUT_SPATIAL,
        "args": [
            "--run", "all",
            "--seed", "42",
            "--split", "spatial",
            "--spatial-val", "robust",
            "--subgraph", "non_isolate",
            "--mcnemar", "A0,A2e",
        ],
    },
}


def recipe_cli(name: str) -> str:
    """Pretty CLI with relative paths (for display / copy-paste)."""
    spec = RECIPES[name]
    parts = [
        "python -m src.train.physagg",
        "--curated",
        DATA.relative_to(ROOT).as_posix(),
        "--graph-dir",
        GRAPH_DIR.relative_to(ROOT).as_posix(),
        "--output",
        spec["output"].relative_to(ROOT).as_posix(),
        *spec["args"],
    ]
    return " ".join(parts)


def recipe_command(name: str) -> list[str]:
    spec = RECIPES[name]
    return [
        sys.executable,
        "-m",
        "src.train.physagg",
        "--curated",
        str(DATA),
        "--graph-dir",
        str(GRAPH_DIR),
        "--output",
        str(spec["output"]),
        *spec["args"],
    ]


def run_recipe(name: str) -> Path:
    print("$", recipe_cli(name))
    if not RUN_TRAINING:
        print("[dry-run] set RUN_TRAINING = True to execute")
        return RECIPES[name]["output"]
    RECIPES[name]["output"].mkdir(parents=True, exist_ok=True)
    subprocess.run(recipe_command(name), cwd=ROOT, check=True)
    return RECIPES[name]["output"]


for name in RECIPES:
    print("===", name, "===")
    print(recipe_cli(name))
    print()

=== temporal_ablation ===
python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/temporal_noniso --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate --mcnemar A2e,A0 --with-rf

=== spatial_holdout ===
python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/spatial_suchiapa --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate --mcnemar A0,A2e



In [4]:
# Launch (no-op when RUN_TRAINING is False)
out_temporal = run_recipe("temporal_ablation")
out_spatial = run_recipe("spatial_holdout")

$ python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/temporal_noniso --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate --mcnemar A2e,A0 --with-rf
[dry-run] set RUN_TRAINING = True to execute
$ python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/spatial_suchiapa --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate --mcnemar A0,A2e
[dry-run] set RUN_TRAINING = True to execute


## 3. Load metric artefacts

After a local run, `summary.json` and `metrics_*.json` appear under
`experiments/outputs/` (gitignored).

In [5]:
def load_summary(out_dir: Path):
    path = out_dir / "summary.json"
    if not path.exists():
        print("Missing:", path.relative_to(ROOT).as_posix())
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload.get("summary") or []
    return pd.DataFrame(rows)


def load_seed_metrics(out_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(out_dir.glob("metrics_*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        m = obj.get("metrics") or {}
        rows.append(
            {
                "file": path.name,
                "run": obj.get("run"),
                "seed": obj.get("seed"),
                "pr_auc": m.get("pr_auc"),
                "f1": m.get("f1"),
            }
        )
    return pd.DataFrame(rows)


print("Temporal summary")
sum_t = load_summary(out_temporal)
display(sum_t if sum_t is not None else pd.DataFrame({"note": ["run temporal_ablation first"]}))

print("Temporal per-seed metrics")
seed_t = load_seed_metrics(out_temporal)
display(seed_t if not seed_t.empty else pd.DataFrame({"note": ["no metrics_*.json yet"]}))

print("Spatial summary")
sum_s = load_summary(out_spatial)
display(sum_s if sum_s is not None else pd.DataFrame({"note": ["run spatial_holdout first"]}))

Temporal summary
Missing: experiments/outputs/physagg_power/temporal_noniso/summary.json


,note
0,run temporal_ablation first


Temporal per-seed metrics


,note
0,no metrics_*.json yet


Spatial summary
Missing: experiments/outputs/physagg_power/spatial_suchiapa/summary.json


,note
0,run spatial_holdout first


## Direct CLI (same recipes)

```bash
# from repository root
python -m src.graphs.build --input data/datos_wifi_curated_anon.parquet --output data/graphs

python -m src.train.physagg \
  --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate \
  --mcnemar A2e,A0 --with-rf \
  --output experiments/outputs/physagg_power/temporal_noniso

python -m src.train.physagg \
  --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate \
  --mcnemar A0,A2e \
  --output experiments/outputs/physagg_power/spatial_suchiapa
```